In [1]:
import pandas as pd
import plotly.express as px
import plotly.subplots as sp

# Análisis de los datos (EDA) y selección de un subdataset para el proyecto

Dado el volumen de datos y las limitantes viculadas con tiempos de procesamiento y costo computacional de textos, vamos a buscar recortar el dataset en lo referido tanto al periodo de tiempo, como a los stock tickers a utilizar en el análisis.

In [2]:
df = pd.read_csv('nasdaq_subdataset.csv')
df.head()

,Date,Article_title,Stock_symbol,Url,Author,Article
0,2023-12-16 23:00:00 UTC,Interesting A Put And Call Options For August ...,A,https://www.nasdaq.com/articles/interesting-a-...,NaN,"Investors in Agilent Technologies, Inc. (Symbo..."
1,2023-12-12 00:00:00 UTC,Wolfe Research Initiates Coverage of Agilent T...,A,https://www.nasdaq.com/articles/wolfe-research...,NaN,"Fintel reports that on December 13, 2023, Wolf..."
2,2023-12-12 00:00:00 UTC,Agilent Technologies Reaches Analyst Target Price,A,https://www.nasdaq.com/articles/agilent-techno...,NaN,"In recent trading, shares of Agilent Technolog..."
3,2023-12-07 00:00:00 UTC,Agilent (A) Enhances BioTek Cytation C10 With ...,A,https://www.nasdaq.com/articles/agilent-a-enha...,NaN,Agilent Technologies A is enhancing its BioTek...
4,2023-12-07 00:00:00 UTC,"Pre-Market Most Active for Dec 7, 2023 : SQQQ,...",A,https://www.nasdaq.com/articles/pre-market-mos...,NaN,The NASDAQ 100 Pre-Market Indicator is up 70.2...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2491778 entries, 0 to 2491777
Data columns (total 6 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Date           object 
 1   Article_title  object 
 2   Stock_symbol   object 
 3   Url            object 
 4   Author         float64
 5   Article        object 
dtypes: float64(1), object(5)
memory usage: 114.1+ MB


In [4]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df.Date.min(), df.Date.max()

(Timestamp('2009-04-08 00:00:00+0000', tz='UTC'),
 Timestamp('2024-01-09 00:00:00+0000', tz='UTC'))

In [5]:
df.Stock_symbol.value_counts()

Stock_symbol
BROGW    10456
BPYPO     9979
BHFAL     9614
T         9449
PMAY      9108
         ...  
BPYPP        1
BPYPN        1
VRMEW        1
ABEQ         1
GOODN        1
Name: count, Length: 4694, dtype: int64

## Gráfico: cantidad de noticias/mes

In [6]:
# Agrupamos por año-mes
df_counts = df.groupby(df['Date'].dt.to_period('M')).size().reset_index(name='count')

# Convertimos el período a datetime para que Plotly lo interprete bien
df_counts['Date'] = df_counts['Date'].dt.to_timestamp()

# Grafico
fig = px.bar(
    df_counts,
    x='Date',
    y='count',
    title='Cantidad de noticias por mes',
    labels={'Date': 'Mes', 'count': 'Cantidad de noticias'},
    template='plotly_white'
)

fig.update_traces(marker_color="#eb990c")

# Nos aseguramos de que el gráfico muestre un ticks por año en el eje x
fig.update_xaxes(
    dtick="M12",
    tickformat="%Y",
    ticklabelmode="period"
)

fig.show()


C:\Users\Ceci\AppData\Local\Temp\ipykernel_12248\105019702.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_counts = df.groupby(df['Date'].dt.to_period('M')).size().reset_index(name='count')


## Gráfico: stock symbols más frecuentes

In [7]:
col_symbol = 'Stock_symbol'
top_n = 15 # (None para mostrar todos)

# Agrupamos por ticket
df_counts = df.groupby(col_symbol).size().reset_index(name='count')
df_counts = df_counts.sort_values('count', ascending=False)

if top_n is not None:
    df_counts = df_counts.head(top_n)

# Gráfico de barras horizontales
fig = px.bar(
    df_counts,
    x='count',
    y=col_symbol,
    orientation='h',
    title=f'Cantidad de noticias por símbolo{" (Top " + str(top_n) + ")" if top_n else ""}',
    labels={'count': 'Cantidad de noticias', col_symbol: 'Símbolo'},
    text='count',
    template='plotly_white'
)

# Estilo visual
fig.update_traces(
    marker_color="#eb990c"

)

# Ajustes para legibilidad
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=400 + 20 * len(df_counts),
    xaxis_title='Cantidad de noticias',
    yaxis_title='Símbolo',
    font=dict(size=12)
)

fig.show()

## Distribución temporal de noticias por ticker

Si bien probamos diversos gráficos, por el volumen de datos nos costaba visualizar la información para hacer la selección de período de tiempo y los tickers a usar.

Lo que buscábamos era una ventana de tiempo de 3-6 meses con noticias prácticamente diarias para cada ticker seleccionado.

La alternativa que encontramos fue construir una tabla de doble entrada con los días en las filas y los tickers en las columnas, que luego transpusimos y exportamos a un google sheet para su análisis visual.

De gráficos previos, habíamos visto que las noticias cubrían mayor cantidad de los tickers de mayor volumnen entre 2017 y 2023, por lo que decidimos filtrar el dataset a ese rango de años.

In [8]:
#Filtramos para quedarnos con los años entre 2017-2023
df_filtered = df.copy()

df_filtered = df[(df_filtered['Date'].dt.year >= 2017) & 
                            (df_filtered['Date'].dt.year <= 2023)].copy()

#Creamos la columna Año-Mes como string
df_filtered['YearMonth'] = df_filtered['Date'].dt.strftime('%Y-%m')

#Creamos la lista de todos los dias en el rango de fechas de df_filtered
fechas_completas = pd.date_range(df_filtered['Date'].min(), df_filtered['Date'].max(), freq='D')
meses_completos = pd.Series(fechas_completas).dt.strftime('%Y-%m').unique()

#Guardamos los "meses completos" en el DataFrame de salida 
df_result = pd.DataFrame({'YearMonth': meses_completos})

#Contar días sin artículos por mes para cada acción

symbols = df_filtered['Stock_symbol'].unique().tolist()

for sym in symbols:
    df_sym = df_filtered[df_filtered['Stock_symbol'] == sym].copy()
    # Contamos artículos por día
    articulos_por_dia = df_sym.groupby('Date').size()
    # Reindexamos para que los días sin artículos aparezcan con 0
    articulos_por_dia = articulos_por_dia.reindex(fechas_completas, fill_value=0)
    # Convertimos a DataFrame para agrupar por YearMonth
    df_dias = articulos_por_dia.reset_index()
    df_dias.columns = ['Date', 'count']
    df_dias['YearMonth'] = df_dias['Date'].dt.strftime('%Y-%m')
    # Contamos los días sin artículos por mes
    dias_sin_articulos = df_dias.groupby('YearMonth').apply(lambda x: (x['count'] == 0).sum())
    # Unimos esta info al df_result (df de salida)
    df_result[sym] = df_result['YearMonth'].map(dias_sin_articulos)

df_result


C:\Users\Ceci\AppData\Local\Temp\ipykernel_12248\3250624845.py:32: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\Ceci\AppData\Local\Temp\ipykernel_12248\3250624845.py:32: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\Ceci\AppData\Local\Temp\ipykernel_12248\3250624845.py:32: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version o

,YearMonth,A,AA,AAAU,AACG,AADR,AAL,AAMC,AAME,AAN,...,ZOM,ZROZ,ZS,ZTO,ZTR,ZTS,ZUMZ,ZUO,ZYME,ZYXI
0,2017-01,31,31,31,31,29,10,31,31,25,...,31,30,31,28,30,29,25,31,31,31
1,2017-02,28,28,28,28,25,13,28,28,21,...,28,28,28,25,26,21,13,28,28,28
2,2017-03,31,31,31,31,29,13,31,31,30,...,31,30,31,30,29,26,24,31,31,31
3,2017-04,30,30,30,30,30,12,30,29,28,...,30,30,30,27,28,25,27,30,28,30
4,2017-05,31,31,31,31,29,13,30,31,21,...,31,31,31,27,30,20,29,31,31,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,2023-08,19,24,31,30,31,7,30,31,25,...,31,30,13,21,31,22,30,23,25,30
80,2023-09,20,22,30,30,30,11,27,30,25,...,30,29,11,25,29,23,26,28,30,29
81,2023-10,24,20,30,31,31,8,31,31,24,...,31,30,13,24,31,18,31,30,31,29
82,2023-11,20,22,29,30,30,12,30,30,24,...,30,29,14,22,30,24,27,26,27,28


In [9]:
dft = df_result.transpose()

In [10]:
dft.head()

,0,1,2,3,4,5,6,7,8,9,...,74,75,76,77,78,79,80,81,82,83
YearMonth,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,...,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12
A,31,28,31,30,31,30,31,31,30,31,...,16,26,13,22,25,19,20,24,20,28
AA,31,28,31,30,31,30,31,31,30,31,...,15,17,19,22,18,24,22,20,22,25
AAAU,31,28,31,30,31,30,31,31,30,31,...,30,29,31,30,31,31,30,30,29,29
AACG,31,28,31,30,31,30,31,31,30,31,...,31,30,31,30,31,30,30,31,30,31


In [11]:
dft.to_csv('nasdaq_dias_sin_articulos_T.csv')

Analizando el archivo nasdaq_dias_sin_articulos_T encontramos un cluster de días entre en 1-8-23 y el 30-11-23, donde para el siguiente listado de acciones tenemos para la gran mayoría de los días al menos una noticia diaria.

Listado stock symbols: [AAPL,AMD,AMZN,BRK,BSMRCGRO,DIS,DNOVEAOA,F,FDEV,FFEB,FSMB,GDMA,GOOG,GS,
GSEE,HCRB,KJUL,KO,MSFT,NVDA,PFFL,PMAY,SPY,TSLA,UBER,UCIB,UFEB,UOCT,WINC,WLDR,WMT,WTRE]

Es por que seleccionamos estos stock symbols y este período de tiempo para el proyecto.

[Google Sheet con el detalle](https://docs.google.com/spreadsheets/d/1lZpl7PRT4rCnjdrdEax2ISs0Wi3IM6aQMkQwBB_QQnE/edit?gid=1628539407#gid=1628539407)

# EDA del subdataset seleccionado y preparación final

In [12]:
# Rango de fechas
inicio = "2023-08-01"
fin = "2023-11-30"

# Lista de símbolos
symbols = [
    "AAPL", "AMD", "AMZN", "BRK", "BSMRCGRO", "DIS", "DNOVEAOA", "F", "FDEV",
    "FFEB", "FSMB", "GDMA", "GOOG", "GS", "GSEE", "HCRB", "KJUL", "KO", "MSFT",
    "NVDA", "PFFL", "PMAY", "SPY", "TSLA", "UBER", "UCIB", "UFEB", "UOCT",
    "WINC", "WLDR", "WMT", "WTRE"
]

df_proyecto = df[
    (df["Date"].between(inicio, fin)) &
    (df["Stock_symbol"].isin(symbols))
].copy()

# Ver resultado
print(df_proyecto.info())
df_proyecto.head()

<class 'pandas.core.frame.DataFrame'>
Index: 125261 entries, 12323 to 2412904
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   Date           125261 non-null  datetime64[ns, UTC]
 1   Article_title  125261 non-null  object             
 2   Stock_symbol   125261 non-null  object             
 3   Url            125261 non-null  object             
 4   Author         0 non-null       float64            
 5   Article        125261 non-null  object             
dtypes: datetime64[ns, UTC](1), float64(1), object(4)
memory usage: 6.7+ MB
None


,Date,Article_title,Stock_symbol,Url,Author,Article
12323,2023-11-30 00:00:00+00:00,UK antitrust regulator wins appeal over Apple ...,AAPL,https://www.nasdaq.com/articles/uk-antitrust-r...,NaN,Adds details from ruling and CMA comment in pa...
12324,2023-11-30 00:00:00+00:00,Japan aircon king Daikin looks to custom chips...,AAPL,https://www.nasdaq.com/articles/japan-aircon-k...,NaN,"By Sam Nussey and Miho Uranaka\nTOKYO, Dec 1 (..."
12325,2023-11-30 00:00:00+00:00,Judge set to rule on Berkshire Hathaway reques...,AAPL,https://www.nasdaq.com/articles/judge-set-to-r...,NaN,By Jonathan Stempel\nNov 30 (Reuters) - A Dela...
12326,2023-11-30 00:00:00+00:00,Analysts predict more brands will flee X after...,AAPL,https://www.nasdaq.com/articles/analysts-predi...,NaN,By Chavi Mehta and Jaspreet Singh\nNov 30 (Reu...
12327,2023-11-30 00:00:00+00:00,Netflix (NFLX) to Offer Grand Theft Auto Trilo...,AAPL,https://www.nasdaq.com/articles/netflix-nflx-t...,NaN,Netflix NFLX has announced its intention to in...


In [13]:
df_proyecto.drop(columns=['Author'], inplace=True)

df_proyecto.reset_index()

df_proyecto.head(2)

,Date,Article_title,Stock_symbol,Url,Article
12323,2023-11-30 00:00:00+00:00,UK antitrust regulator wins appeal over Apple ...,AAPL,https://www.nasdaq.com/articles/uk-antitrust-r...,Adds details from ruling and CMA comment in pa...
12324,2023-11-30 00:00:00+00:00,Japan aircon king Daikin looks to custom chips...,AAPL,https://www.nasdaq.com/articles/japan-aircon-k...,"By Sam Nussey and Miho Uranaka\nTOKYO, Dec 1 (..."


In [14]:
df_proyecto.to_csv('nasdaq_subdataset_proyecto.csv', index=False)

## Grafico: cantidad de noticias por día en el subdataset seleccionado

In [32]:
col_symbol = 'Stock_symbol'
top_n = None

# Agrupamos por ticket
df_counts = df_proyecto.groupby(col_symbol).size().reset_index(name='count')
df_counts = df_counts.sort_values('count', ascending=False)

if top_n is not None:
    df_counts = df_counts.head(top_n)

# Gráfico de barras horizontales
fig = px.bar(
    df_counts,
    x='count',
    y=col_symbol,
    orientation='h',
    title=f'Cantidad de noticias por símbolo{" (Top " + str(top_n) + ")" if top_n else ""}',
    labels={'count': 'Cantidad de noticias', col_symbol: 'Símbolo'},
    text='count',
    template='plotly_white'
)

# Estilo visual
fig.update_traces(
    marker_color="#eb990c"

)

# Ajustes para legibilidad
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=400 + 20 * len(df_counts),
    xaxis_title='Cantidad de noticias',
    yaxis_title='Símbolo',
    font=dict(size=12)
)

fig.show()

## Grafico: cantidad de noticias por día por stock symbol en el subdataset seleccionado

In [39]:
# Agrupamos por día y por símbolo
df_counts = (
    df_proyecto
    .groupby([df_proyecto['Date'].dt.to_period('D'), 'Stock_symbol'])
    .size()
    .reset_index(name='count')
)

# Convertimos el período a datetime para que Plotly lo interprete bien
df_counts['Date'] = df_counts['Date'].dt.to_timestamp()

# Ordenamos por símbolo y fecha para consistencia
df_counts = df_counts.sort_values(['Stock_symbol', 'Date'])

# Lista de símbolos únicos
symbols = df_counts['Stock_symbol'].unique()

# Generamos Subplots: uno por símbolo
fig = sp.make_subplots(
    rows=len(symbols),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{s}" for s in symbols]
)

# Generamos un gráfico de barras por símbolo
for i, sym in enumerate(symbols, start=1):
    df_sym = df_counts[df_counts['Stock_symbol'] == sym]

    fig.add_bar(
        x=df_sym['Date'],
        y=df_sym['count'],
        name=sym,
        marker_color="#eb990c",
        row=i,
        col=1
    )

    # Eje X: ticks mensuales y etiquetas visibles
    fig.update_xaxes(
        type="date",
        showticklabels=True,
        ticks="outside",
        tickformat="%d-%m-%Y",
        #dtick="M1",
        tickangle=45,
        row=i,
        col=1
    )

# Layout final
fig.update_layout(
    height=250 * len(symbols),
    showlegend=False,
    template="plotly_white",
    title="Cantidad de noticias por día (por símbolo)"
)

fig.show()

C:\Users\Ceci\AppData\Local\Temp\ipykernel_12248\2661888542.py:4: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

